# Fake News Detection using Random Forest

Final Year Project 1 (FYP 1)  
Student: IAIMAN ILYAS BIN AZMI  
ID: 23B07I008

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [2]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ilyas\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ilyas\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\ilyas\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [3]:
import pandas, numpy, sklearn, nltk, joblib

print("Pandas:", pandas.__version__)
print("NumPy:", numpy.__version__)
print("Scikit-learn:", sklearn.__version__)
print("NLTK:", nltk.__version__)
print("Joblib:", joblib.__version__)

Pandas: 2.3.3
NumPy: 2.3.5
Scikit-learn: 1.7.2
NLTK: 3.9.2
Joblib: 1.5.2


In [4]:
fake_df = pd.read_csv("dataset/Fake.csv")
true_df = pd.read_csv("dataset/True.csv")

print("Fake shape:", fake_df.shape)
print("True shape:", true_df.shape)
fake_df.head(2), true_df.head(2)


Fake shape: (23481, 4)
True shape: (21417, 4)


(                                               title  \
 0   Donald Trump Sends Out Embarrassing New Year’...   
 1   Drunk Bragging Trump Staffer Started Russian ...   
 
                                                 text subject  \
 0  Donald Trump just couldn t wish all Americans ...    News   
 1  House Intelligence Committee Chairman Devin Nu...    News   
 
                 date  
 0  December 31, 2017  
 1  December 31, 2017  ,
                                                title  \
 0  As U.S. budget fight looms, Republicans flip t...   
 1  U.S. military to accept transgender recruits o...   
 
                                                 text       subject  \
 0  WASHINGTON (Reuters) - The head of a conservat...  politicsNews   
 1  WASHINGTON (Reuters) - Transgender people will...  politicsNews   
 
                  date  
 0  December 31, 2017   
 1  December 29, 2017   )

In [5]:
# Add labels
fake_df["label"] = 1   # Fake news
true_df["label"] = 0   # Real news

# Combine datasets
df = pd.concat([fake_df, true_df], ignore_index=True)

# Check combined dataset
print("Combined shape before cleaning:", df.shape)
print(df["label"].value_counts())

# Check missing values
print("\nMissing values:")
print(df.isnull().sum())

Combined shape before cleaning: (44898, 5)
label
1    23481
0    21417
Name: count, dtype: int64

Missing values:
title      0
text       0
subject    0
date       0
label      0
dtype: int64


In [6]:
# Combine title and text into one column
df["content"] = df["title"].fillna("") + " " + df["text"].fillna("")

# Check duplicates before removing
print("Duplicates before removing:", df.duplicated(subset=["content"]).sum())

# Remove duplicates
df = df.drop_duplicates(subset=["content"])

# Check dataset after removing duplicates
print("Shape after removing duplicates:", df.shape)
print(df["label"].value_counts())

Duplicates before removing: 5793
Shape after removing duplicates: (39105, 6)
label
0    21197
1    17908
Name: count, dtype: int64


In [7]:
# Check missing values
print("Missing values after cleaning:")
print(df.isnull().sum())


Missing values after cleaning:
title      0
text       0
subject    0
date       0
label      0
content    0
dtype: int64


In [8]:
import re

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def clean_text(text):
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    
    # Remove punctuation & numbers
    text = re.sub(r"[^a-z\s]", "", text)
    
    # Tokenize
    words = text.split()
    
    # Remove stopwords & lemmatize
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    
    return " ".join(words)


In [9]:
# Apply preprocessing to a small sample (for demo & testing)
#df["clean_text"] = df["text"].apply(clean_text)
df["clean_text"] = df["content"].apply(clean_text)

#df[["text", "clean_text"]].head(3)
df[["content", "clean_text"]].head(3)


,content,clean_text
0,Donald Trump Sends Out Embarrassing New Year’...,donald trump sends embarrassing new year eve m...
1,Drunk Bragging Trump Staffer Started Russian ...,drunk bragging trump staffer started russian c...
2,Sheriff David Clarke Becomes An Internet Joke...,sheriff david clarke becomes internet joke thr...


In [10]:
X = df["clean_text"]
y = df["label"]

print("X sample:", X.iloc[0][:200])
print("y sample:", y.iloc[0])


X sample: donald trump sends embarrassing new year eve message disturbing donald trump wish american happy new year leave instead give shout enemy hater dishonest fake news medium former reality show star one j
y sample: 1


In [11]:
# Split data
X = df["clean_text"]
y = df["label"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

Train size: 27373
Test size: 11732


In [12]:
# TF-IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF train shape:", X_train_tfidf.shape)
print("TF-IDF test shape:", X_test_tfidf.shape)

TF-IDF train shape: (27373, 5000)
TF-IDF test shape: (11732, 5000)


In [13]:
from sklearn.ensemble import RandomForestClassifier

# Create model
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train model
rf_model.fit(X_train_tfidf, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [14]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Predict
y_pred = rf_model.predict(X_test_tfidf)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Classification report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# Confusion matrix
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9969314694851688

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      6359
           1       1.00      1.00      1.00      5373

    accuracy                           1.00     11732
   macro avg       1.00      1.00      1.00     11732
weighted avg       1.00      1.00      1.00     11732


Confusion Matrix:

[[6348   11]
 [  25 5348]]


In [15]:
import joblib

# Save model
joblib.dump(rf_model, "random_forest_model.pkl")

# Save TF-IDF vectorizer
joblib.dump(tfidf, "tfidf_vectorizer.pkl")

print("Model and vectorizer saved successfully!")

Model and vectorizer saved successfully!
